In [1]:
#!pip install langchain==0.2.14
#!pip install langchain_community==0.2.14
#!pip install pydantic==1.10.8
#!pip install gradio==4.44.1
!pip install -U openai langchain langchain_community gradio ipywidgets faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.2.7-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_text_splitters-1.1.0-py3-none-any.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.9 MB/s  0:00:00
Using cached langchain-1.2.6-py3-none-any.whl (108 kB)
Using cached langchain_core-1.2.7-py3-none-any.whl (490 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_text_splitters-1.1.0-py3-none-any.whl (34 kB)
  Attempting uninstall: openai
    Found existing installation: openai 1.109.1
    Uninstalling openai-1.109.1:
      Successfully uninstalled openai-1.109.1
  Attempting uninstall: langchain-core━━━━━━━━━━ 0/5 [openai]
    Found existing installation: langchain-core 0.3.830/5 [openai]
    Uninstalling l

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.pdf', multiple=True)
display(uploader)

FileUpload(value=(), accept='.pdf', description='Upload', multiple=True)

In [6]:
print(uploader.value[0])

{'name': '소나기.pdf', 'type': 'application/pdf', 'size': 123544, 'content': <memory at 0x7f88a146b940>, 'last_modified': datetime.datetime(2026, 1, 19, 2, 29, 53, 962000, tzinfo=datetime.timezone.utc)}


In [7]:
bytes_data = uploader.value[0].content.tobytes()

In [8]:
print(type(bytes_data))

<class 'bytes'>


In [9]:
import os

def save_uploaded_pdf(uploader):
    uploaded_file = list(uploader.value)[0]
    file_name = uploaded_file['name']
    save_path = os.path.join(os.getcwd(), file_name)
    with open(save_path, 'wb') as f:
        f.write(uploaded_file['content'])
    
    print(f'Uploaded and saved: {file_name}')
    return file_name

file_name = save_uploaded_pdf(uploader)

Uploaded and saved: 소나기.pdf


In [10]:
!pip install pypdf

Defaulting to user installation because normal site-packages is not writeable


In [11]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_name)
pages = loader.load_and_split()

I0000 00:00:1768956856.584810   13466 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [12]:
pages

[Document(metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.11131', 'creationdate': '2021-11-29T20:05:50+09:00', 'author': 'MYHOME', 'moddate': '2021-11-29T20:05:50+09:00', 'pdfversion': '1.4', 'source': '소나기.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='- 1 -\n소나기황순원소년은 개울가에서 소녀를 보자 곧 윤 초시네 증손녀(曾孫女)딸이라는 걸 알 수 있었다. 소녀는 개울에다 손을 잠그고 물장난을 하고 있는 것이다. 서울서는 이런 개울물을 보지 못하기나 한 듯이.벌써 며칠째 소녀는, 학교에서 돌아오는 길에 물장난이었다. 그런데, 어제까지 개울 기슭에서 하더니, 오늘은 징검다리 한가운데 앉아서 하고 있다.소년은 개울둑에 앉아 버렸다. 소녀가 비키기를 기다리자는 것이다.요행 지나가는 사람이 있어, 소녀가 길을 비켜 주었다.다음 날은 좀 늦게 개울가로 나왔다.이 날은 소녀가 징검다리 한가운데 앉아 세수를 하고 있었다. 분홍 스웨터 소매를 걷어올린 목덜미가 마냥 희었다.한참 세수를 하고 나더니, 이번에는 물 속을 빤히 들여다 본다. 얼굴이라도 비추어 보는 것이리라. 갑자기 물을 움켜 낸다. 고기 새끼라도 지나가는 듯.소녀는 소년이 개울둑에 앉아 있는 걸 아는지 모르는지 그냥 날쌔게 물만 움켜 낸다. 그러나, 번번이 허탕이다. 그대로 재미있는 양, 자꾸 물만 움킨다. 어제처럼 개울을 건너는 사람이 있어야 길을 비킬 모양이다.그러다가 소녀가 물 속에서 무엇을 하나 집어 낸다. 하얀 조약돌이었다. 그리고는 벌떡 일어나 팔짝팔짝 징검다리를 뛰어 건너간다.다 건너가더니만 홱 이리로 돌아서며,“이 바보.”조약돌이 날아왔다.소년은 저도 모르게 벌떡 일어섰다.단발 머리를 나풀거리며 소녀가 막 달린다

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

# PDF 로드
loader = PyPDFLoader(file_name)
documents = loader.load()

# 텍스트 분할
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)
docs = text_splitter.split_documents(documents)

len(docs)


7

In [15]:
docs[:3]

[Document(metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.11131', 'creationdate': '2021-11-29T20:05:50+09:00', 'author': 'MYHOME', 'moddate': '2021-11-29T20:05:50+09:00', 'pdfversion': '1.4', 'source': '소나기.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='- 1 -\n소나기황순원소년은 개울가에서 소녀를 보자 곧 윤 초시네 증손녀(曾孫女)딸이라는 걸 알 수 있었다. 소녀는 개울에다 손을 잠그고 물장난을 하고 있는 것이다. 서울서는 이런 개울물을 보지 못하기나 한 듯이.벌써 며칠째 소녀는, 학교에서 돌아오는 길에 물장난이었다. 그런데, 어제까지 개울 기슭에서 하더니, 오늘은 징검다리 한가운데 앉아서 하고 있다.소년은 개울둑에 앉아 버렸다. 소녀가 비키기를 기다리자는 것이다.요행 지나가는 사람이 있어, 소녀가 길을 비켜 주었다.다음 날은 좀 늦게 개울가로 나왔다.이 날은 소녀가 징검다리 한가운데 앉아 세수를 하고 있었다. 분홍 스웨터 소매를 걷어올린 목덜미가 마냥 희었다.한참 세수를 하고 나더니, 이번에는 물 속을 빤히 들여다 본다. 얼굴이라도 비추어 보는 것이리라. 갑자기 물을 움켜 낸다. 고기 새끼라도 지나가는 듯.소녀는 소년이 개울둑에 앉아 있는 걸 아는지 모르는지 그냥 날쌔게 물만 움켜 낸다. 그러나, 번번이 허탕이다. 그대로 재미있는 양, 자꾸 물만 움킨다. 어제처럼 개울을 건너는 사람이 있어야 길을 비킬 모양이다.그러다가 소녀가 물 속에서 무엇을 하나 집어 낸다. 하얀 조약돌이었다. 그리고는 벌떡 일어나 팔짝팔짝 징검다리를 뛰어 건너간다.다 건너가더니만 홱 이리로 돌아서며,“이 바보.”조약돌이 날아왔다.소년은 저도 모르게 벌떡 일어섰다.단발 머리를 나풀거리며 소녀가 막 달린다

In [16]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

In [20]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 임베딩 모델
embeddings = OpenAIEmbeddings()

# 이미 만든 docs 사용
# docs = text_splitter.split_documents(documents)

# FAISS 인덱스 생성
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

# 로컬 저장
vectorstore.save_local("shower")


In [21]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [22]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
다음 문서를 참고해서 질문에 답하세요.

문서:
{context}

질문:
{question}

답변:
""")


In [23]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | chat
)


In [24]:
response = rag_chain.invoke("소년은 어디에서 처음 소녀를 만났나?")
print(response.content)


소년은 개울가에서 처음 소녀를 만났습니다.


In [26]:
response = rag_chain.invoke("소년은 소녀에게 무엇을 주려고 했나?")
print(response.content)

소년은 소녀에게 호두를 맛보여 주고 싶어 했습니다. 그는 덕쇠 할아버지네 호두밭에서 호두를 따고, 소녀에게 맛보여야겠다는 생각을 하였습니다.


In [27]:
response = rag_chain.invoke("소녀는 소년에게 무엇을 주었나?")
print(response.content)

소녀는 소년에게 하얀 조약돌을 주었습니다. 소녀가 물속에서 조약돌을 집어 낸 후, 징검다리를 뛰어 건너며 "이 바보"라고 말하며 조약돌을 소년에게 던졌습니다.


In [28]:
response = rag_chain.invoke("소녀와 윤초시는 어떤 관계인가?")
print(response.content)

소녀는 윤 초시의 증손녀(曾孫女)입니다.
